In [1]:
import time
import socket
from queue import Queue 
from threading import Thread, Condition
from ipywidgets import widgets
from IPython.display import display
import random
import statistics

In [2]:
class EXP_Output:
    def __init__(self):
        #self.out = widgets.Output(layout={'border': '1px solid black'})
        #display(self.out)
        #with self.out:
        #print("Experiment outputs")

        self.Queue_Print = Queue()
        self.thread = Thread(target=self.loop_Print)
        self.thread.start()

    def print(self, msg):
        text_out = time.strftime("%H:%M:%S", time.localtime()) + "> "
        self.Queue_Print.put(text_out+msg)

    def loop_Print(self):
        while True:
            msg = self.Queue_Print.get()
            #with self.out:
            print(msg)
            self.Queue_Print.task_done()

In [3]:
class ALE_TextInput:
    
    def __init__(self, exp_out):
        
        self.Queue_User = Queue()
        self.exp_out = exp_out

        # Create a text input widget
        self.Text = widgets.Text(
                        value='',
                        placeholder='Type something and press enter',
                        description='To send:',
                        disabled=False
                    )

        # create a button object
        self.Botton =  widgets.Button(
                            description='Submit',
                            disabled=False,
                            button_style='', 
                            tooltip='Click me to submit a string',
                        )

        # Define a function to handle the click action on the button
        def on_submit_button_clicked(b):

            global thread_running
            
            msg = self.Text.value
            #self.exp_out.print("ALE_0: received input: "+str(msg))
                
            if thread_running == True:
                if msg == "end":
                    thread_running = False
                
                for i in range(20):
                    self.Queue_User.put(msg+str(i))

            self.Text.value = ''  # Clear the input field after submission

        # Attach the event to the button
        self.Botton.on_click(on_submit_button_clicked)
        
        # Display the widgets
        display(self.Text, self.Botton)
        
    # to get a message from user queue. this function can block the thread
    def get(self):
        return self.Queue_User.get()

In [4]:
class ALE_TR:
    
    def __init__(self, name, upper_Tx, lower_TR, exp_out):
        
        self.Name     = name
        self.Upper_Tx = upper_Tx  # must provide get()
        self.Lower_TR = lower_TR  # must provide send(msg), and receive()
        self.exp_out  = exp_out
        
    def loop_Tx(self):

        global thread_running
        c = 0
        self.exp_out.print(self.Name + ": loop_Tx starting")
    
        while (thread_running == True):

            c = c + 1

            # get text from Upper_Tx, which must provide a get method
            # this thread is blocked here
            msg = self.Upper_Tx.get()
            
            text_out = self.Name + " Tx: message " + str(c) + ": " + str(msg)
            self.exp_out.print(text_out)

            # add the text to the queue
            self.Lower_TR.send(msg)
            
    def loop_Rx(self):
        
        global thread_running
        c = 0
        self.exp_out.print(self.Name + ": loop_Rx starting")
    
        while (thread_running == True):

            c = c + 1

            # get message from a lower layer
            # this thread is blocked here
            msg = self.Lower_TR.receive()
            
            text_out = self.Name + " Rx: message " + str(c) + ": " + str(msg)
            self.exp_out.print(text_out)

In [5]:
# state of DLE
STATE_READY_TO_SEND = 0 # ready to send a packet to the DLE entity in another node through lower layer
STATE_WAITING_ACK   = 1 # waiting for ACK from the DLE entity in another node

# events of DLE
EVENT_UPPER_TX  = 0 # upper layer wants to send a packet
EVENT_LOWER_DAT = 1 # lower layer forwards an incoming data packet
EVENT_LOWER_ACK = 2 # lower layer forwards an incoming data packet
EVENT_TIMEOUT   = 3 # timer timeout
        
class DLE_TR_FSM:
    
    def __init__(self, name, lower_TR, exp_out):
        
        self.Name      = name
        self.Lower_TR  = lower_TR   # must provide send(msg) and receive()
        self.Queue_Tx  = Queue()
        self.Queue_Rx  = Queue()
        self.exp_out   = exp_out
        
        # create a queue for Finite State Machine
        self.Queue_FSM = Queue()
        
        # create a flag = condition to start sending
        self.cv_TxOp = Condition()
        self.TxOp    = True

        # init state to STATE_WAITING_UPPER
        self.State = STATE_READY_TO_SEND
        
        self.Procedure = [[self.FSM_upper_Tx, self.FSM_abnormal],
                          [self.FSM_lower_Rx, self.FSM_lower_Rx],
                          [self.FSM_abnormal, self.FSM_lower_Rx_ack],
                          [self.FSM_abnormal, self.FSM_timeout]] 
        

        # new features for retransmission
        # timer
        self.cv_Timer      = Condition()
        self.timer_active  = False
        self.timer_counter = 0
        
        # transmission 
        self.tx_buffer  = ''
        self.tx_seq_num = '0'
        
        # reception
        self.tx_ack_num = '0'
        
        
    def loop_Tx(self):

        global thread_running
        c = 0
        self.exp_out.print(self.Name + ": loop_Tx starting")
    
        while (thread_running == True):

            # waiting for the ready to sent signal
            with self.cv_TxOp: 
                while (self.TxOp != True): 
                    self.cv_TxOp.wait()
            
                self.TxOp = False
                
                c = c + 1

                # get a message from queue
                msg = self.Queue_Tx.get()
                text_out = self.Name + " Tx: message " + str(c) + ": " + str(msg)
                self.exp_out.print(text_out)
            
                self.event_add(EVENT_UPPER_TX, msg) 

    def loop_Rx(self):
        
        global thread_running
        c = 0
        self.exp_out.print(self.Name + ": loop_Rx starting")
    
        while (thread_running == True):

            c = c + 1

            # get message from a lower layer
            # this thread is blocked here
            msg = self.Lower_TR.receive()
            
            text_out = self.Name + " Rx: message " + str(c) + ": " + str(msg)
            self.exp_out.print(text_out)
            
            # add an event for FSM, assuming that msg is a sequence of bytes
            msg_type = msg[0]
            if (msg_type == '0'):
                self.event_add(EVENT_LOWER_DAT, msg[1:])
            elif (msg_type == '1'):
                self.event_add(EVENT_LOWER_ACK, msg[1:])
            else:
                text_out = self.Name + " Rx: message type unknown " + msg_type
                self.exp_out.print(text_out)

    def receive(self):
        return self.Queue_Rx.get()
    
    def send(self, msg):
        self.Queue_Tx.put(msg)
        
    def event_add(self, ev_type, msg):
        
        # preparing an event
        if (isinstance(msg, str)):
            msg = msg.encode()
        
        # the event is a sequence of bytes
        event = ev_type.to_bytes(1, "big")+msg
        
        # add event to queue
        self.Queue_FSM.put(event)        
        

    def loop_FSM(self):
        
        global thread_running
        self.exp_out.print(self.Name + ": loop_FSM starting")
        
        while (thread_running == True):
            
            # get the next event
            event = self.Queue_FSM.get()
            
            ev_type = event[0]
            
            text_out = self.Name + " FSM: state: " + str(self.State)+" event type: " + str(ev_type)
            self.exp_out.print(text_out)
            
            # process the event
            msg = event[1:].decode('utf-8') # message becomes a string
            self.Procedure[ev_type][self.State](msg)
        
    def FSM_abnormal(self, msg):
        
        text_out = self.Name + " FSM: error! " + msg
        self.exp_out.print(text_out)
    
    def FSM_upper_Tx(self, msg):   
        
        text_out = self.Name + " FSM: to send: " + msg
        self.exp_out.print(text_out)

        # buffer the message (for retransmission)
        self.tx_buffer = msg
        
        # prepare to send a data message
        msg = '0' + self.tx_seq_num + msg     # inicating a new packet and its sequence number
        self.Lower_TR.send(msg)

        # waiting for ACK
        self.State = STATE_WAITING_ACK
        
        # start timer
        with self.cv_Timer:
            self.timer_active  = True
            self.timer_counter = 10
            self.cv_Timer.notify()    
    
    def FSM_lower_Rx(self, msg):

        # get the sequence number
        seq = msg[0]
        msg = msg[1:]
        
        # received a new data packet        
        text_out = self.Name + " FSM: received: " + msg
        self.exp_out.print(text_out)

        if (seq == self.tx_ack_num):
            
            # put the message in a receiving queue
            self.Queue_Rx.put(msg)
            
            self.tx_ack_num = chr(ord(self.tx_ack_num) + 1)
            if (self.tx_ack_num == chr(ord('9') + 1)):
                self.tx_ack_num = '0'
        
        else:
            text_out  = self.Name + " FSM: received received frame " + seq
            text_out += " but expected " + self.tx_ack_num
            self.exp_out.print(text_out)
            

        # to send an ACK
        self.Lower_TR.send('1'+seq)

                
    def FSM_lower_Rx_ack(self, msg):
        
        ack = msg[0] # get the ack number
        if (ack != self.tx_seq_num):
            
            # the received ack does not match the seq
            text_out = "ACK # " + ack + " does not match local seq # " + self.tx_seq_num
            self.exp_out.print(text_out)
            return

        # stop timer
        self.timer_active = False
        
        # inc the seq
        self.tx_seq_num = chr(ord(self.tx_seq_num) + 1)
        if (self.tx_seq_num == chr(ord('9') + 1)):
                self.tx_seq_num = '0'
        
        # ready to send the next upper layer packet
        with self.cv_TxOp:
            self.State = STATE_READY_TO_SEND
            self.TxOp  = True
            self.cv_TxOp.notify()
            
    def FSM_timeout(self, msg):
        
        text_out = self.Name + " FSM: to resend frame " + self.tx_seq_num + " " + self.tx_buffer
        self.exp_out.print(text_out)
        
        # prepare to send a data message
        msg = '0' + self.tx_seq_num + self.tx_buffer     # inicating a new packet and its sequence number
        self.Lower_TR.send(msg)
        
        # start timer
        with self.cv_Timer:
            self.timer_active  = True
            self.timer_counter = 10
            self.cv_Timer.notify()

    def loop_timer(self):
        
        global thread_running
        self.exp_out.print(self.Name + ": loop_Timer starting")
        
        while (thread_running == True):
            
            with self.cv_Timer: 
                while (self.timer_active == False): 
                    self.cv_Timer.wait() 
            
                time.sleep(0.5)

                if (self.timer_counter == 0):
                    # add an event for timeout
                    self.event_add(EVENT_TIMEOUT, 'x')
                    self.timer_active = False

                else:
                    self.timer_counter = self.timer_counter - 1

In [6]:
class PLE_TR:

    def __init__(self, name, Socket, AP_Tx, AP_Rx, exp_out):
        
        self.Name      = name
        self.Socket    = Socket
        self.AP_Tx     = AP_Tx
        self.AP_Rx     = AP_Rx
        self.Queue_Tx  = Queue()
        self.Queue_Rx  = Queue()
        self.exp_out   = exp_out
        
    def loop_Tx(self):
        
        global thread_running
        self.exp_out.print(self.Name + ": loop_Tx starting")
    
        while (thread_running == True):
        
            # get a message from queue
            msg = self.Queue_Tx.get()
            
            text_out = self.Name + " Tx: message: " + str(msg)
            self.exp_out.print(text_out)
            
            #NEW Code
            #time.sleep(random.random()*2)  


            # sending the message using socket
            msg_bytes = str.encode(msg)

            # #HERE
            # if msg.startswith("1"):  # ACK starts with 1
            #     if random.random() < 1.0:  # 100% chance to lose ACK (for testing)
            #         self.exp_out.print("ACK loss simulated")
            #         continue  # Skip sending ACK

            if (random.random() < 0.1):
                self.exp_out.print("transmission lost")
            
            
            else: #new line code
            #self.Socket.sendto(msg_bytes, self.AP_Tx)
            
                if (random.random() < 0.05):
                    self.exp_out.print("transmission duplicate")
            #else:
                # sending the message using socket
                    self.Socket.sendto(msg_bytes, self.AP_Tx)
            
                self.Socket.sendto(msg_bytes, self.AP_Tx) #new line code

    # def loop_Rx(self):
        
    #     global thread_running
    #     global bufferSize
    #     self.exp_out.print(self.Name + ": loop_Rx starting")
    
    #     # binding the socket with the IP and port
    #     self.Socket.bind(self.AP_Rx)
        
    #     while (thread_running == True):
        
    #         # get a message from socket, this thread is blocked here
    #         msg_addr = self.Socket.recvfrom(bufferSize)

    #         # to emulate a slow receiver
    #         time.sleep(0.5)
    
    #         msg  = msg_addr[0].decode('utf-8')
    #         addr = msg_addr[1]
            
    #         text_out = self.Name + " Rx: from " + str(addr) + ": " + str(msg)
    #         self.exp_out.print(text_out)
            
    #         self.Queue_Rx.put(msg)


    def loop_Rx(self):
        global thread_running
        global bufferSize
        self.exp_out.print(self.Name + ": loop_Rx starting")

        self.Socket.bind(self.AP_Rx)

        while (thread_running == True):
            # Wait for message (blocking)
            msg_addr = self.Socket.recvfrom(bufferSize)
    
            #  Add random delay between 0 and 2 seconds
            delay = random.uniform(0, 2)
            time.sleep(delay)
    
            msg  = msg_addr[0].decode('utf-8')
            addr = msg_addr[1]
    
            text_out = self.Name + f" Rx (after {delay:.2f}s delay): from {addr}: {msg}"
            self.exp_out.print(text_out)
    
            self.Queue_Rx.put(msg)
            
    def send(self, msg):
        self.Queue_Tx.put(msg)
        
    def receive(self):
        return self.Queue_Rx.get()

In [7]:
thread_running = False
bufferSize = 1024

# (0) creating an output object
exp_out = EXP_Output()

# (1) create physical layer entities
AP_local_1  = ("127.0.0.1", 30000)
AP_remote_1 = ("127.0.0.1", 31111)
Socket_1    = socket.socket(family=socket.AF_INET, type=socket.SOCK_DGRAM)
PLE_1       = PLE_TR("PLE_Alice", Socket_1, AP_remote_1, AP_local_1, exp_out)

# (2) create date link layer entities
DLE_1 = DLE_TR_FSM("DLE_Alice", PLE_1, exp_out)

# (3) create application layer entities
ALE_0 = ALE_TextInput(exp_out)
ALE_1 = ALE_TR("ALE_Alice", ALE_0, DLE_1, exp_out)

Text(value='', description='To send:', placeholder='Type something and press enter')

Button(description='Submit', style=ButtonStyle(), tooltip='Click me to submit a string')

19:33:49> ALE_Alice: loop_Tx starting
19:33:49> ALE_Alice: loop_Rx starting
19:33:49> DLE_Alice: loop_Tx starting
19:33:49> DLE_Alice: loop_Rx starting
19:33:49> PLE_Alice: loop_Tx starting
19:33:49> PLE_Alice: loop_Rx starting
19:33:49> ALE_Bob: loop_Tx starting
19:33:49> ALE_Bob: loop_Rx starting
19:33:49> DLE_Bob: loop_Tx starting
19:33:49> DLE_Bob: loop_Rx starting
19:33:49> PLE_Bob: loop_Tx starting
19:33:49> PLE_Bob: loop_Rx starting
19:33:49> DLE_Alice: loop_FSM starting
19:33:49> DLE_Bob: loop_FSM starting
19:33:49> DLE_Alice: loop_Timer starting
19:33:49> DLE_Bob: loop_Timer starting
19:34:09> ALE_Alice Tx: message 1: Alice to Bob0
19:34:09> ALE_Alice Tx: message 2: Alice to Bob1
19:34:09> ALE_Alice Tx: message 3: Alice to Bob2
19:34:09> ALE_Alice Tx: message 4: Alice to Bob3
19:34:09> ALE_Alice Tx: message 5: Alice to Bob4
19:34:09> ALE_Alice Tx: message 6: Alice to Bob5
19:34:09> ALE_Alice Tx: message 7: Alice to Bob6
19:34:09> ALE_Alice Tx: message 8: Alice to Bob7
19:34:09

In [8]:
# (4) create physical layer entities
AP_local_2  = ("127.0.0.1", 31111)
AP_remote_2 = ("127.0.0.1", 30000)
Socket_2    = socket.socket(family=socket.AF_INET, type=socket.SOCK_DGRAM)
PLE_2       = PLE_TR("PLE_Bob", Socket_2, AP_remote_2, AP_local_2, exp_out)

# (5) create date link layer entities
DLE_2 = DLE_TR_FSM("DLE_Bob", PLE_2, exp_out)

# (6) create application layer entities
ALE_3 = ALE_TextInput(exp_out)
ALE_2 = ALE_TR("ALE_Bob", ALE_3, DLE_2, exp_out)

Text(value='', description='To send:', placeholder='Type something and press enter')

Button(description='Submit', style=ButtonStyle(), tooltip='Click me to submit a string')

In [9]:
# start the loops of all entities
# all loops must be blocked at a certain position

t1_1 = Thread(target = ALE_1.loop_Tx, args = ()) 
t2_1 = Thread(target = ALE_1.loop_Rx, args = ()) 
t3_1 = Thread(target = DLE_1.loop_Tx, args = ())
t4_1 = Thread(target = DLE_1.loop_Rx, args = ())
t5_1 = Thread(target = PLE_1.loop_Tx, args = ()) 
t6_1 = Thread(target = PLE_1.loop_Rx, args = ())
f1_1 = Thread(target = DLE_1.loop_FSM, args = ())
f2_1 = Thread(target = DLE_1.loop_timer, args = ())


t1_2 = Thread(target = ALE_2.loop_Tx, args = ()) 
t2_2 = Thread(target = ALE_2.loop_Rx, args = ()) 
t3_2 = Thread(target = DLE_2.loop_Tx, args = ())
t4_2 = Thread(target = DLE_2.loop_Rx, args = ())
t5_2 = Thread(target = PLE_2.loop_Tx, args = ()) 
t6_2 = Thread(target = PLE_2.loop_Rx, args = ()) 
f1_2 = Thread(target = DLE_2.loop_FSM, args = ())
f2_2 = Thread(target = DLE_2.loop_timer, args = ())

thread_running = True

t1_1.start()
t2_1.start()
t3_1.start()
t4_1.start()
t5_1.start()
t6_1.start()
t1_2.start()
t2_2.start()
t3_2.start()
t4_2.start()
t5_2.start()
t6_2.start()
f1_1.start()
f1_2.start()
f2_1.start()
f2_2.start()

In [ ]:
# class MLE_TR(PLE_TR):
#     def __init__(self, name, Socket, AP_Tx, AP_Rx, exp_out):
#         super().__init__(name, Socket, AP_Tx, AP_Rx, exp_out)

#         # Create the button using ipywidgets
#         self.button =  widgets.Button(
#                             description='Measure',
#                             disabled=False,
#                             button_style='', 
#                             tooltip='Click me to measure and send 100 messages',
#                         )
        
#         def buttonClick(button):
#             # This button sends 100 messages using a lower layer interface 
#             for i in range(100):
#                 # Create a packet with experiment ID, message ID and the current time with nanosecond accuracy using time.time_ns() function
#                 experimentID = 1
#                 messageID = i
#                 timestamp = time.time_ns()
#                 packet = f"Exp:{experimentID}, Msg:{messageID}, Time:{timestamp}"
                
#                 # Send the packet through the bottom layer interface 
#                 self.send(packet)
                
#                 # Generate Experiment Output (Sec 4 Part 5 & 6)
#                 bob_mle.measurementOutputGenerator()

#         self.button.on_click(buttonClick)

#         display(self.button)
        
#         # Variables that will receive packets and duplicate packets
#         self.receivedPacket = []
#         self.duplicatedPacketSet = set()

    
#     def loop_Rx(self):
#         global thread_running
#         global bufferSize

#         # binding the socket with the IP and port
#         self.Socket.bind(self.AP_Rx)

#         while (thread_running == True):
#             # get a message from socket, this thread is blocked here
#             msg_addr = self.Socket.recvfrom(bufferSize)

#             msg  = msg_addr[0]
#             addr = msg_addr[1]

#             print(time.strftime("%H:%M:%S", time.localtime()), end=' ')
#             print("%s Rx from %s: %s"%(self.Name, addr, msg))
#             self.Queue_Rx.put(msg)

#             # Process the received packet
#             packet_parts = msg.decode().split(", ")
#             experimentID = int(packet_parts[0].split(":")[1])
#             messageID = int(packet_parts[1].split(":")[1])
#             sentTimenNS = int(packet_parts[2].split(":")[1])
            
#             # Verify and add to the list if the packet is duplicate or not in the packet list
#             # This is done for Part 4 (Section 4 Part 4)
#             if (experimentID, messageID) not in self.receivedPacket:
#                 self.receivedPacket.append((experimentID, messageID, sentTimenNS, time.time_ns()))
#             else:
#                 self.duplicatedPacketSet.add((experimentID, messageID))
                
#     # Analyzes lost, duplicated packets and delay to return the required result in experiments
#     def analyzer(self):
#         delays = []
#         lostPackets = 0
#         duplicatedPacketSet = 0

#         for i in range(100):
#             successfulPackets = [p for p in self.receivedPacket if p[1] == i]

#             if len(successfulPackets) == 0:
#                 lostPackets += 1
#             else:
#                 sentTimenNS = successfulPackets[0][2]
#                 receivedTimeNS = successfulPackets[0][3]
#                 delays.append((receivedTimeNS - sentTimenNS) / 1e6)  

#                 if len(successfulPackets) > 1:
#                     duplicatedPacketSet += len(successfulPackets) - 1
        
#         #Store Results
#         if len(delays) > 0:
#             averageDelay = statistics.mean(delays)
#         else:
#             averageDelay = 0

#         averageLossRate = lostPackets / 100
#         averageDuplicationRate = duplicatedPacketSet / 100

#         results = { 'averageDelay': averageDelay, 'averageLossRate': averageLossRate, 'averageDuplicationRate': averageDuplicationRate }

#         return results
    
#     # Function in charge of monitoring the process of receiving packets   
#     def measurementOutputGenerator(self, packetCount=90, delaySeconds=10):
#         #Start Timer
#         startTimer = time.time()
        
#         # Check if a minimum of packets received has been met or time has been passed
#         while True:
#             elapsedTime = time.time() - startTimer
#             successPacketCount = len(self.receivedPacket)
            
#             if successPacketCount >= packetCount or elapsedTime >= delaySeconds:
#                 break
            
#             # Wait for 1 second before checking again
#             time.sleep(1)
            
#         #Analyzes the results of packet receipt, saves it and returns it once called
#         results = self.analyzer()
#         self.resultsOutput(results)
#         return results
    
#     #Prints out the results
#     def resultsOutput(self, results):
#         print("Measurement Output:")
#         print(f"Average delay (ms): {results['averageDelay']}")
#         print(f"Average message loss rate: {results['averageLossRate']}")
#         print(f"Average message duplication rate: {results['averageDuplicationRate']}")
    
# exp_out = EXP_Output()

# #Tester Entities (MLE)        
# AP_local_mle_1 = ("127.0.0.1", 32000)
# AP_local_mle_2 = ("127.0.0.1", 32111)
# AP_remote_mle_2 = ("127.0.0.1", 32000)
# AP_remote_mle_1 = ("127.0.0.1", 32111)


# # Create new sockets for alice_mle and bob_mle
# alice_mle_socket = socket.socket(family=socket.AF_INET, type=socket.SOCK_DGRAM)
# bob_mle_socket = socket.socket(family=socket.AF_INET, type=socket.SOCK_DGRAM)

# # Call class MLE_TR
# alice_mle = MLE_TR("MLE_Alice", alice_mle_socket, AP_remote_mle_1, AP_local_mle_1, exp_out)
# bob_mle = MLE_TR("MLE_Bob", bob_mle_socket, AP_remote_mle_2, AP_local_mle_2, exp_out)

# DLE_3 = DLE_TR_FSM("DLE_Alice", alice_mle, exp_out)
# DLE_4 = DLE_TR_FSM("DLE_Bob", bob_mle, exp_out)

# # start the loops of all entities
# # all loops must be blocked at a certain position
# alice_mle_thread_tx = Thread(target=alice_mle.loop_Tx, args = ())
# alice_mle_thread_rx = Thread(target=alice_mle.loop_Rx, args = ())
# bob_mle_thread_tx = Thread(target=bob_mle.loop_Tx, args = ())
# bob_mle_thread_rx = Thread(target=bob_mle.loop_Rx, args = ())

# # Start threads for MLE_Alice and MLE_Bob
# thread_running = True

# alice_mle_thread_tx.start()
# alice_mle_thread_rx.start()
# bob_mle_thread_tx.start()
# bob_mle_thread_rx.start()

# time.sleep(1)

# # Run the generator
# bob_mle.measurementOutputGenerator()

In [ ]:
# class MLE_TR(PLE_TR):
#     def __init__(self, name, Socket, AP_Tx, AP_Rx, exp_out, experiment_id = 1):
#         super().__init__(name, Socket, AP_Tx, AP_Rx, exp_out)
#         self.experiment_id = experiment_id
#         self.received_packets = []
#         self.duplicate_packets = set()
        
#         self.button = widgets.Button(description=f'Run Measurment for {name}')
#         self.button.on_click(self.send_messages_callback)
#         display(self.button)

#     def send_messages_callback(self, b):
#         """
#         Callback function triggered by the button click.
#         Sends 100 messages with an unique ID and timesamp
#         """

#         for msg_id in range(1, 101):
#             timestamp = time.time_ns()
#             packet = f"{self.experiment_id}|{msg_id}|{timestamp}" # Construct the package format
#             self.send(packet)            
            
#         print(f"Button Pressed")

#     # TODO do meausements function

#     def loop_Rx(self):
#         """
#         Loop that continously receives messages from the lower layer
#         For each received packet, it:
#             - Parses the packet to extract the data
#             - Calculates the delay between when the package was sent and received
#             - Tracks duplicates received packets

#         Once 100 unique messages are received, it calcules:
#             - The average delay
#             - The loss rate (if missing, 20% chance of that happening)
#             - Duplicate rate, if any
#         """
#         global thread_running
#         global bufferSize
    
#         self.Socket.bind(self.AP_Rx)
        
#         delays = []  # Initialize the delays list to store delay values

#         while (thread_running == True):

#             message_address = self.Socket.recvfrom(bufferSize)

#             message = message_address[0]
#             address = message_address[1]

#             try:
#                 experiment_id, message_id, sent_time = message.decode().split('|')
#                 message_id = int(message_id)
#                 sent_time = int(sent_time)
#                 print(time.strftime("%H:%M:%S", time.localtime()), end=' ')
#                 print("%s Rx from %s: %s"%(self.Name, address, message))
#                 self.Queue_Rx.put(message)
#             except Exception as e:
#                 print(f"Error - {e} - parsing packet: ", message)
#                 continue
                
#             # Verify and add to the list if the packet is duplicate or not in the packet list
#             # This is done for Part 4 (Section 4 Part 4)
#             else:
#                 received_time = time.time_ns()
#                 self.received_packets.append((experiment_id, message_id, sent_time, received_time))
#                 delays.append((received_time - sent_time) / 1e6)  # Calculate delay in milliseconds and store it
#             else:
#                 self.received_packets.append((experiment_id, message_id, sent_time, time.time_ns()))

#         def display_computations():
#             """
#             Compute the average delay, loss rate, and duplicate rate
#             """
#             # Calculate average delay across all received messages
#             avg_delay = sum(delays) / len(delays) if delays else 0
            
#             # Compute loss rate based on missing unique message IDs
#             loss_rate = (total_expected - len(self.received_messages)) / total_expected
            
#             # Compute duplicate rate by summing up extra receptions beyond the first for each message
#             duplicate_count = sum(count - 1 for count in self.received_messages.values() if count > 1)
#             duplicate_rate = duplicate_count / total_expected
            
#             print(f"Average Delay: {avg_delay:.2f} ms")
#             print(f"Loss Rate: {loss_rate * 100:.2f}%")
#             print(f"Duplicate Rate: {duplicate_rate * 100:.2f}%")

#         # Calculate average delay across all received messages
#         avg_delay = sum(delays) / len(delays) if delays else 0
        
#         # Compute loss rate based on missing unique message IDs
#         loss_rate = (total_expected - len(self.received_messages)) / total_expected
        
#         # Compute duplicate rate by summing up extra receptions beyond the first for each message
#         duplicate_count = sum(count - 1 for count in self.received_messages.values() if count > 1)
#         duplicate_rate = duplicate_count / total_expected
        
#         print(f"Average Delay: {avg_delay:.2f} ms")
#         print(f"Loss Rate: {loss_rate * 100:.2f}%")
#         print(f"Duplicate Rate: {duplicate_rate * 100:.2f}%")

In [ ]:
class MLE_TR(PLE_TR):
    def __init__(self, name, Socket, AP_Tx, AP_Rx, exp_out, experiment_id = 1):
        super().__init__(name, Socket, AP_Tx, AP_Rx, exp_out)
        self.experiment_id = experiment_id
        self.received_packets = []
        self.received_counts = {}
        
        self.button = widgets.Button(description=f'Run {name}')
        self.button.on_click(self.send_messages_callback)
        display(self.button)

    def send_messages_callback(self, b):

        for msg_id in range(1, 101):
            timestamp = time.time_ns()
            packet = f"{self.experiment_id}|{msg_id}|{timestamp}"
            self.send(packet)            
            
        self.exp_out.print(f"Button Pressed")

    def display_computations(self):

        total_expected = 100
        delays = [(received_time - sent_time) / 1e6 for (_, _, sent_time, received_time) in self.received_packets]
        avg_delay = sum(delays) / len(delays) if delays else 0
        
        unique_received = len(self.received_counts)
        loss_rate = (total_expected - unique_received) / total_expected
        
        duplicate_count = sum(1 for count in self.received_counts.values() if count > 1)
        duplicate_rate = duplicate_count / total_expected
        
        self.exp_out.print(f"--- Measurement Results for {self.Name} ---")
        self.exp_out.print(f"Average Delay: {avg_delay:.2f} ms")
        self.exp_out.print(f"Loss Rate: {loss_rate * 100:.2f}%")
        self.exp_out.print(f"Duplicate Rate: {duplicate_rate * 100:.2f}%")

    def loop_Rx(self):

        global thread_running
        global bufferSize
    
        self.Socket.bind(self.AP_Rx)
        
        while (thread_running == True):

            message_address = self.Socket.recvfrom(bufferSize)

            message = message_address[0]
            address = message_address[1]

            try:
                experiment_id, message_id, sent_time = message.decode().split('|')
                message_id = int(message_id)
                sent_time = int(sent_time)
                print(time.strftime("%H:%M:%S", time.localtime()), end=' ')
                print("%s Rx from %s: %s"%(self.Name, address, message))
                self.Queue_Rx.put(message)
            except Exception as e:
                self.exp_out.print(f"Error - {e} - parsing packet: {message}")
                continue

            key = message_id
            if key in self.received_counts:
                self.received_counts[key]+= 1
            else:
                self.received_counts[key] = 1
                self.received_packets.append((experiment_id, message_id, sent_time, time.time_ns()))
            
            self.display_computations()


exp_out = EXP_Output()

AP_local_MLE_Alice = ("localhost", 32000)
AP_Local_MLE_Bob = ("localhost", 32222)
AP_Remote_MLE_Bob = ("localhost", 32000)
AP_remote_MLE_Alice = ("localhost", 32222)

alice_MLE_socket = socket.socket(family=socket.AF_INET, type=socket.SOCK_DGRAM)
bob_MLE_socket = socket.socket(family=socket.AF_INET, type=socket.SOCK_DGRAM)

# Call class MLE_TR
alice_MLE = MLE_TR("MLE_Alice", alice_MLE_socket, AP_remote_MLE_Alice, AP_local_MLE_Alice, exp_out)
bob_MLE = MLE_TR("MLE_Bob", bob_MLE_socket, AP_Remote_MLE_Bob, AP_Local_MLE_Bob, exp_out)


alice_thread_tx = Thread(target=alice_MLE.loop_Tx)
alice_thread_rx = Thread(target=alice_MLE.loop_Rx)
bob_thread_tx = Thread(target=bob_MLE.loop_Tx)
bob_thread_rx = Thread(target=bob_MLE.loop_Rx)

# Start threads for MLE_Alice and MLE_Bob
thread_running = True

alice_thread_tx.start()
alice_thread_rx.start()
bob_thread_tx.start()
bob_thread_rx.start()

Button(description='Run MLE_Alice', style=ButtonStyle())

Button(description='Run MLE_Bob', style=ButtonStyle())

19:34:47> MLE_Alice: loop_Tx starting
19:34:47> MLE_Bob: loop_Tx starting


19:34:56> Button Pressed
19:34:56> MLE_Alice Tx: message: 1|1|1743118496780947100
19:34:56 MLE_Bob Rx from ('127.0.0.1', 32000): b'1|1|1743118496780947100'
19:34:56> --- Measurement Results for MLE_Bob ---
19:34:56> Average Delay: 2.18 ms
19:34:56> Loss Rate: 99.00%
19:34:56> Duplicate Rate: 0.00%
19:34:56> MLE_Alice Tx: message: 1|2|1743118496780947100
19:34:56 MLE_Bob Rx from ('127.0.0.1', 32000): b'1|2|1743118496780947100'
19:34:56> MLE_Alice Tx: message: 1|3|1743118496780947100
19:34:56> --- Measurement Results for MLE_Bob ---
19:34:56> Average Delay: 2.68 ms
19:34:56> Loss Rate: 98.00%
19:34:56> Duplicate Rate: 0.00%
19:34:56 MLE_Bob Rx from ('127.0.0.1', 32000): b'1|3|1743118496780947100'
19:34:56> MLE_Alice Tx: message: 1|4|1743118496780947100
19:34:56> --- Measurement Results for MLE_Bob ---
19:34:56> Average Delay: 2.84 ms
19:34:56> Loss Rate: 97.00%
19:34:56> Duplicate Rate: 0.00%
19:34:56 MLE_Bob Rx from ('127.0.0.1', 32000): b'1|4|1743118496780947100'
19:34:56> MLE_Alice Tx